In [0]:
# =============================================================
# NOTEBOOK 04 — ML TRAINING + MLFLOW
# Quick Commerce Dark Store Intelligence System
# Layer: ML (Train, Evaluate, Track, Register)
# Source: gold_instacart database
# =============================================================

In [0]:
import mlflow
import mlflow.spark
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import col

GOLD_DB = "gold_instacart"

# Load Gold tables
df_customers = spark.table(f"{GOLD_DB}.customer_features")
df_products  = spark.table(f"{GOLD_DB}.product_features")

print("✅ Gold tables loaded")
print(f"   customers → {df_customers.count():,} rows")
print(f"   products  → {df_products.count():,} rows")

✅ Gold tables loaded
   customers → 206,209 rows
   products  → 49,685 rows


In [0]:
print("Customer feature columns:")
print(df_customers.columns)

print("\nChurn distribution:")
df_customers.groupBy("is_churning").count().show()

Customer feature columns:
['user_id', 'total_orders', 'avg_days_between_orders', 'avg_basket_size', 'reorder_rate', 'max_order_number', 'avg_order_hour', 'is_churning', 'ingested_at']

Churn distribution:
+-----------+------+
|is_churning| count|
+-----------+------+
|          0| 65486|
|          1|140723|
+-----------+------+



In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col

# Define feature columns
feature_cols = [
    "total_orders",
    "avg_days_between_orders",
    "avg_basket_size",
    "reorder_rate",
    "max_order_number",
    "avg_order_hour"
]

# Drop nulls
df_ml = df_customers.select(feature_cols + ["is_churning"]).dropna()

# Assemble features into single vector
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_ml = assembler.transform(df_ml).select("features", "is_churning") \
        .withColumnRenamed("is_churning", "label")

# Train/Test split — 80/20
df_train, df_test = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"✅ Features assembled")
print(f"   Training rows : {df_train.count():,}")
print(f"   Testing rows  : {df_test.count():,}")
print(f"   Feature count : {len(feature_cols)}")

✅ Features assembled
   Training rows : 164,801
   Testing rows  : 41,408
   Feature count : 6


In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Set experiment name
mlflow.set_experiment("/quick_commerce_churn")

# Define evaluators
binary_evaluator = BinaryClassificationEvaluator(metricName="areaUnderROC")
multi_evaluator  = MulticlassClassificationEvaluator(metricName="f1")

# ── RUN 1 — LOGISTIC REGRESSION ─────────────────────
with mlflow.start_run(run_name="LogisticRegression_baseline"):

    # Define model (remove weightCol - not supported)
    lr = LogisticRegression(
        featuresCol="features",
        labelCol="label",
        maxIter=10,
        regParam=0.01
    )

    # Train
    lr_model = lr.fit(df_train)

    # Predict
    lr_predictions = lr_model.transform(df_test)

    # Evaluate
    auc = binary_evaluator.evaluate(lr_predictions)
    f1  = multi_evaluator.evaluate(lr_predictions)

    # Log to MLflow
    mlflow.log_param("model",    "LogisticRegression")
    mlflow.log_param("maxIter",  10)
    mlflow.log_param("regParam", 0.01)
    mlflow.log_metric("auc_roc", round(auc, 4))
    mlflow.log_metric("f1_score", round(f1, 4))

    print("=" * 45)
    print("RUN 1 — LOGISTIC REGRESSION")
    print("=" * 45)

RUN 1 — LOGISTIC REGRESSION


In [0]:
from pyspark.ml.classification import RandomForestClassifier

# ── RUN 2 — RANDOM FOREST ───────────────────────────
with mlflow.start_run(run_name="RandomForest_default"):

    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        numTrees=100,
        maxDepth=10,
        seed=42
    )

    # Train
    rf_model = rf.fit(df_train)

    # Predict
    rf_predictions = rf_model.transform(df_test)

    # Evaluate
    auc = binary_evaluator.evaluate(rf_predictions)
    f1  = multi_evaluator.evaluate(rf_predictions)

    # Log to MLflow
    mlflow.log_param("model",    "RandomForest")
    mlflow.log_param("numTrees", 100)
    mlflow.log_param("maxDepth", 10)
    mlflow.log_metric("auc_roc",  round(auc, 4))
    mlflow.log_metric("f1_score", round(f1, 4))

    print("=" * 45)
    print("RUN 2 — RANDOM FOREST")
    print("=" * 45)
    print(f"  AUC-ROC  : {round(auc, 4)}")
    print(f"  F1 Score : {round(f1, 4)}")
    print("=" * 45)

RUN 2 — RANDOM FOREST
  AUC-ROC  : 0.9999
  F1 Score : 0.9979


In [0]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# ── RUN 3 — TUNED RANDOM FOREST ─────────────────────
with mlflow.start_run(run_name="RandomForest_tuned"):

    rf_tuned = RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        numTrees=200,
        maxDepth=15,
        minInstancesPerNode=2,
        seed=42
    )

    # Train
    rf_tuned_model = rf_tuned.fit(df_train)

    # Predict
    rf_tuned_predictions = rf_tuned_model.transform(df_test)

    # Evaluate
    auc = binary_evaluator.evaluate(rf_tuned_predictions)
    f1  = multi_evaluator.evaluate(rf_tuned_predictions)

    # Feature importance
    importances = rf_tuned_model.featureImportances
    feature_importance = list(zip(feature_cols, importances))
    feature_importance.sort(key=lambda x: x[1], reverse=True)

    # Log to MLflow
    mlflow.log_param("model",    "RandomForest_tuned")
    mlflow.log_param("numTrees", 200)
    mlflow.log_param("maxDepth", 15)
    mlflow.log_param("minInstancesPerNode", 2)
    mlflow.log_metric("auc_roc",  round(auc, 4))
    mlflow.log_metric("f1_score", round(f1, 4))

    # Log feature importances
    for feat, imp in feature_importance:
        mlflow.log_metric(f"importance_{feat}", round(float(imp), 4))

    print("=" * 45)
    print("RUN 3 — TUNED RANDOM FOREST")
    print("=" * 45)
    print(f"  AUC-ROC  : {round(auc, 4)}")
    print(f"  F1 Score : {round(f1, 4)}")
    print("\n  Feature Importances:")
    for feat, imp in feature_importance:
        print(f"  {feat:30s} → {round(float(imp), 4)}")
    print("=" * 45)

RUN 3 — TUNED RANDOM FOREST
  AUC-ROC  : 0.9999
  F1 Score : 0.9979

  Feature Importances:
  avg_days_between_orders        → 0.9017
  max_order_number               → 0.0714
  total_orders                   → 0.0157
  reorder_rate                   → 0.0085
  avg_basket_size                → 0.0014
  avg_order_hour                 → 0.0013


In [0]:
from mlflow.models.signature import infer_signature
import pandas as pd

model_name = "quick_commerce_churn_model"

# End any active run first
mlflow.end_run()

# Create sample input from original customer features
sample_input  = df_customers.select(feature_cols).limit(5).toPandas()
sample_output = pd.DataFrame({"prediction": [0.0, 1.0, 1.0, 0.0, 1.0]})

# Infer signature
signature = infer_signature(sample_input, sample_output)

# Register in a fresh run
with mlflow.start_run(run_name="RandomForest_tuned_registered"):
    mlflow.spark.log_model(
        rf_tuned_model,
        "model",
        registered_model_name=model_name,
        dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp_vol",
        signature=signature,
        input_example=sample_input
    )
    mlflow.log_metric("auc_roc",  0.9999)
    mlflow.log_metric("f1_score", 0.9979)
    mlflow.log_param("model",     "RandomForest_tuned")
    mlflow.log_param("numTrees",  200)
    mlflow.log_param("maxDepth",  15)

print("=" * 45)
print("MODEL REGISTERED IN MLFLOW")
print("=" * 45)
print(f"  Model Name : {model_name}")
print(f"  AUC-ROC    : 0.9999")
print(f"  F1 Score   : 0.9979")
print("=" * 45)
print("🏆 Best model registered!")

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/03/09 07:00:01 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==

MODEL REGISTERED IN MLFLOW
  Model Name : quick_commerce_churn_model
  AUC-ROC    : 0.9999
  F1 Score   : 0.9979
🏆 Best model registered!


In [0]:
# Customer Segmentation (KMeans)

In [0]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

mlflow.end_run()

with mlflow.start_run(run_name="KMeans_customer_segmentation"):

    kmeans = KMeans(
        featuresCol="features",
        k=3,          # 3 segments: Loyal, Occasional, At-Risk
        seed=42,
        maxIter=20
    )

    # Train
    kmeans_model = kmeans.fit(df_train)

    # Predict
    kmeans_predictions = kmeans_model.transform(df_test)

    # Evaluate using Silhouette score
    evaluator = ClusteringEvaluator()
    silhouette = evaluator.evaluate(kmeans_predictions)

    # Log to MLflow
    mlflow.log_param("model",   "KMeans")
    mlflow.log_param("k",       3)
    mlflow.log_param("maxIter", 20)
    mlflow.log_metric("silhouette_score", round(silhouette, 4))

    print("=" * 45)
    print("MODEL 3 — KMEANS SEGMENTATION")
    print("=" * 45)
    print(f"  Silhouette Score : {round(silhouette, 4)}")
    print(f"  Segments         : 3")
    print("=" * 45)

    # Show segment distribution
    kmeans_predictions.groupBy("prediction") \
        .count() \
        .withColumnRenamed("prediction", "segment") \
        .orderBy("segment") \
        .show()

MODEL 3 — KMEANS SEGMENTATION
  Silhouette Score : 0.8327
  Segments         : 3
+-------+-----+
|segment|count|
+-------+-----+
|      0|32181|
|      1| 7516|
|      2| 1711|
+-------+-----+



In [0]:
from pyspark.sql.functions import avg, round, col

# Add segment labels to full customer data
df_customers_segmented = kmeans_model.transform(
    assembler.transform(
        df_customers.select(feature_cols + ["user_id", "is_churning"]).dropna()
    )
).select("user_id", "is_churning", "prediction") \
 .withColumnRenamed("prediction", "segment")

# Profile each segment
df_segment_profile = kmeans_model.transform(
    assembler.transform(
        df_customers.select(feature_cols).dropna()
    )
).groupBy("prediction").agg(
    round(avg("total_orders"), 1).alias("avg_orders"),
    round(avg("avg_days_between_orders"), 1).alias("avg_days_between"),
    round(avg("avg_basket_size"), 1).alias("avg_basket"),
    round(avg("reorder_rate"), 2).alias("avg_reorder_rate")
).orderBy("prediction")

print("=" * 55)
print("CUSTOMER SEGMENT PROFILES")
print("=" * 55)
df_segment_profile.show()

CUSTOMER SEGMENT PROFILES
+----------+----------+----------------+----------+----------------+
|prediction|avg_orders|avg_days_between|avg_basket|avg_reorder_rate|
+----------+----------+----------------+----------+----------------+
|         0|      79.6|            14.5|       5.8|            0.39|
|         1|     355.5|            10.4|       8.4|            0.62|
|         2|     901.6|             6.6|      10.3|            0.75|
+----------+----------+----------------+----------+----------------+



In [0]:
from pyspark.sql.functions import when

# Add segment names
df_customers_segmented = df_customers_segmented \
    .withColumn("segment_name",
        when(col("segment") == 0, "At-Risk")
        .when(col("segment") == 1, "Occasional")
        .when(col("segment") == 2, "Loyal"))

# Write to Gold
df_customers_segmented.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_instacart.customer_segments")

print(f"✅ customer_segments → {df_customers_segmented.count():,} rows")

# Summary
df_customers_segmented.groupBy("segment", "segment_name") \
    .count() \
    .orderBy("segment") \
    .show()

print("🏆 Customer Segmentation Complete!")

✅ customer_segments → 206,209 rows
+-------+------------+------+
|segment|segment_name| count|
+-------+------------+------+
|      0|     At-Risk|160087|
|      1|  Occasional| 37553|
|      2|       Loyal|  8569|
+-------+------------+------+

🏆 Customer Segmentation Complete!
